# Adaptive Modulation Using Machine Learning
**Author:** Francesco De Florence

| SNR Range | Modulation | bits/symbol |
|-----------|------------|-------------|
| < 5 dB | BPSK | 1 |
| 5-10 dB | QPSK | 2 |
| 10-20 dB | 16-QAM | 4 |
| > 20 dB | 64-QAM | 6 |


In [ ]:
!pip install numpy pandas matplotlib scikit-learn seaborn scipy -q


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, time, os, json
from scipy.special import erfc
from collections import Counter
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    ConfusionMatrixDisplay, precision_recall_fscore_support)
from sklearn.pipeline import Pipeline
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries loaded!')


## 1. Dataset Generation


In [ ]:
def Q_func(x):
    return 0.5 * erfc(x / np.sqrt(2))

def ber_bpsk(snr_db):
    return Q_func(np.sqrt(2 * 10**(snr_db/10)))

def ber_qpsk(snr_db):
    return Q_func(np.sqrt(2 * 10**(snr_db/10)))

def ber_16qam(snr_db):
    return (3/8) * erfc(np.sqrt(10**(snr_db/10) / 10))

def ber_64qam(snr_db):
    return (7/24) * erfc(np.sqrt(10**(snr_db/10) / 42))

def assign_modulation(snr_db, ber):
    if snr_db < 5 or ber > 1e-2:    return 'BPSK'
    elif snr_db < 10 or ber > 5e-3: return 'QPSK'
    elif snr_db < 20 or ber > 1e-3: return '16-QAM'
    else:                            return '64-QAM'

N = 5000
snr_vals = np.random.uniform(-5, 35, N)
noise    = np.random.normal(0, 0.5, N)
ber_vals = np.clip(ber_bpsk(snr_vals) * (1 + np.abs(noise)), 1e-7, 0.5)
mods     = [assign_modulation(snr_vals[i], ber_vals[i]) for i in range(N)]

df = pd.DataFrame({
    'SNR_dB'    : np.round(snr_vals, 3),
    'BER'       : np.round(ber_vals, 8),
    'log10_BER' : np.round(np.log10(np.clip(ber_vals, 1e-10, 1)), 4),
    'Modulation': mods
})
print(df['Modulation'].value_counts())
print(df.describe().round(4))


## 2. Exploratory Data Analysis


In [ ]:
ORDER  = ['BPSK', 'QPSK', '16-QAM', '64-QAM']
COLORS = {'BPSK':'#E74C3C','QPSK':'#F39C12','16-QAM':'#2ECC71','64-QAM':'#3498DB'}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Adaptive Modulation - EDA', fontsize=14, fontweight='bold')

ax = axes[0,0]
for m in ORDER:
    ax.hist(df[df.Modulation==m]['SNR_dB'], bins=40, alpha=0.6, label=m, color=COLORS[m])
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('Count')
ax.set_title('SNR Distribution per Modulation'); ax.legend()

ax = axes[0,1]
for m in ORDER:
    ax.hist(df[df.Modulation==m]['log10_BER'], bins=40, alpha=0.6, label=m, color=COLORS[m])
ax.set_xlabel('log10(BER)'); ax.set_title('BER Distribution'); ax.legend()

ax = axes[0,2]
for m in ORDER:
    sub = df[df.Modulation==m]
    ax.scatter(sub.SNR_dB, sub.log10_BER, alpha=0.3, s=5, label=m, color=COLORS[m])
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('log10(BER)')
ax.set_title('Feature Space'); ax.legend(markerscale=3)

ax = axes[1,0]
snr_r = np.linspace(-5, 35, 300)
ax.semilogy(snr_r, ber_bpsk(snr_r),  label='BPSK',   color=COLORS['BPSK'],   lw=2)
ax.semilogy(snr_r, ber_qpsk(snr_r),  label='QPSK',   color=COLORS['QPSK'],   lw=2)
ax.semilogy(snr_r, ber_16qam(snr_r), label='16-QAM', color=COLORS['16-QAM'], lw=2)
ax.semilogy(snr_r, ber_64qam(snr_r), label='64-QAM', color=COLORS['64-QAM'], lw=2)
ax.axhline(1e-3, color='gray', ls='--', lw=1, label='BER=1e-3 threshold')
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('BER')
ax.set_title('BER vs SNR Curves'); ax.set_ylim([1e-7, 1]); ax.legend(fontsize=8)

ax = axes[1,1]
counts = df.Modulation.value_counts()[ORDER]
ax.pie(counts, labels=ORDER, autopct='%1.1f%%',
       colors=[COLORS[m] for m in ORDER], startangle=90,
       wedgeprops={'edgecolor':'white','linewidth':2})
ax.set_title('Class Distribution')

ax = axes[1,2]
snr_d = np.linspace(-5, 35, 500)
eff_shannon  = np.log2(1 + 10**(snr_d/10))
eff_adaptive = np.where(snr_d<5, 1, np.where(snr_d<10, 2, np.where(snr_d<20, 4, 6)))
ax.plot(snr_d, eff_shannon,  label='Shannon Limit', color='gray',    lw=2, ls='--')
ax.step(snr_d, eff_adaptive, label='Adaptive Mod',  color='#8E44AD', lw=2.5, where='post')
ax.fill_between(snr_d, eff_adaptive, step='post', alpha=0.15, color='#8E44AD')
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('bits/symbol')
ax.set_title('Spectral Efficiency vs SNR'); ax.legend(); ax.set_ylim([0, 12])

plt.tight_layout(); plt.show()


## 3. Preprocessing & Feature Engineering


In [ ]:
df['SNR_BER_product'] = df.SNR_dB * (-df.log10_BER)
df['SNR_squared']     = df.SNR_dB ** 2
df['BER_log_abs']     = df.log10_BER.abs()

FEATURES = ['SNR_dB', 'log10_BER', 'SNR_BER_product', 'SNR_squared', 'BER_log_abs']
TARGET   = 'Modulation'

X = df[FEATURES].values
y = df[TARGET].values

le    = LabelEncoder()
y_enc = le.fit_transform(y)
print('Classes:', dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.20, random_state=42, stratify=y_enc)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

fig, ax = plt.subplots(figsize=(8, 6))
tmp = df[FEATURES].copy()
tmp['Label'] = y_enc
sns.heatmap(tmp.corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout(); plt.show()


## 4. Model Training


In [ ]:
classifiers = {
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=150, learning_rate=0.1,
                                           max_depth=5, random_state=42))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=10, gamma='scale',
                    probability=True, random_state=42))
    ]),
    'MLP Neural Net': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(128, 64, 32),
                              activation='relu', solver='adam',
                              max_iter=500, random_state=42,
                              early_stopping=True))
    ])
}

results        = {}
trained_models = {}

header = '{:<22} {:>8} {:>8} {:>8} {:>8}'.format('Model','Train','Val','Test','Time')
print(header)
print('-' * 58)

for name, pipe in classifiers.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    elapsed = time.time() - t0
    tr = accuracy_score(y_train, pipe.predict(X_train))
    va = accuracy_score(y_val,   pipe.predict(X_val))
    te = accuracy_score(y_test,  pipe.predict(X_test))
    results[name] = dict(train=tr, val=va, test=te, time=elapsed)
    trained_models[name] = pipe
    row = '{:<22} {:>8.4f} {:>8.4f} {:>8.4f} {:>7.2f}s'.format(name,tr,va,te,elapsed)
    print(row)

best_name  = max(results, key=lambda k: results[k]['val'])
best_model = trained_models[best_name]
print('\nBest model:', best_name, '| val =', round(results[best_name]['val'], 4))


## 5. Model Evaluation


In [ ]:
class_names = le.classes_
y_pred = best_model.predict(X_test)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Model Evaluation - ' + best_name, fontsize=13, fontweight='bold')

# Confusion matrix
ax = axes[0,0]
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix - ' + best_name)

# Per-class metrics
ax = axes[0,1]
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred)
xp = np.arange(len(class_names)); w = 0.25
ax.bar(xp-w, prec, w, label='Precision', color='#3498DB', alpha=0.85)
ax.bar(xp,   rec,  w, label='Recall',    color='#2ECC71', alpha=0.85)
ax.bar(xp+w, f1,   w, label='F1',        color='#E74C3C', alpha=0.85)
ax.set_xticks(xp); ax.set_xticklabels(class_names)
ax.set_ylim([0, 1.12]); ax.set_title('Per-Class Metrics'); ax.legend()

# Accuracy comparison
ax = axes[0,2]
mn = list(results.keys()); short = ['RF','GB','SVM','MLP']
x2 = np.arange(len(mn))
ax.bar(x2-0.2, [results[m]['val']  for m in mn], 0.35, label='Val',  color='#9B59B6', alpha=0.85)
ax.bar(x2+0.2, [results[m]['test'] for m in mn], 0.35, label='Test', color='#1ABC9C', alpha=0.85)
ax.set_xticks(x2); ax.set_xticklabels(short)
ax.set_ylim([0.90, 1.01]); ax.set_title('Accuracy Comparison'); ax.legend()

# Decision boundary 2D
ax = axes[1,0]
sc   = StandardScaler().fit(X[:,:2])
X2   = sc.transform(X[:,:2])
clf2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf2.fit(X2, y_enc)
h = 0.05
xmn, xmx = X2[:,0].min()-0.5, X2[:,0].max()+0.5
ymn, ymx = X2[:,1].min()-0.5, X2[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(xmn,xmx,h), np.arange(ymn,ymx,h))
Z = clf2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.35, cmap=plt.cm.Pastel1)
ax.scatter(X2[:500,0], X2[:500,1], c=y_enc[:500], cmap=plt.cm.Set1, s=8, alpha=0.7)
ax.set_title('Decision Boundary (2D)')
ax.set_xlabel('SNR (norm)'); ax.set_ylabel('log10(BER) (norm)')

# Feature importance
ax = axes[1,1]
imp = trained_models['Random Forest'].named_steps['clf'].feature_importances_
idx = np.argsort(imp)[::-1]
ax.bar(range(len(FEATURES)), imp[idx],
       color=['#3498DB','#E74C3C','#2ECC71','#F39C12','#9B59B6'], alpha=0.85)
ax.set_xticks(range(len(FEATURES)))
ax.set_xticklabels([FEATURES[i] for i in idx], rotation=20, ha='right', fontsize=9)
ax.set_title('Feature Importance (RF)')

# Training time
ax = axes[1,2]
times = [results[m]['time'] for m in mn]
ax.barh(short, times, color=['#3498DB','#E74C3C','#2ECC71','#F39C12'], alpha=0.85)
ax.set_title('Training Time (s)')

plt.tight_layout(); plt.show()
print(classification_report(y_test, y_pred, target_names=class_names))


## 6. Hyperparameter Tuning (GridSearchCV)


In [ ]:
if best_name == 'Random Forest':
    param_grid = {'clf__n_estimators': [100, 200, 300],
                  'clf__max_depth': [None, 10, 20],
                  'clf__max_features': ['sqrt', 'log2']}
elif best_name == 'Gradient Boosting':
    param_grid = {'clf__n_estimators': [100, 200],
                  'clf__learning_rate': [0.05, 0.1, 0.2],
                  'clf__max_depth': [3, 5, 7]}
elif best_name == 'SVM (RBF)':
    param_grid = {'clf__C': [1, 10, 100],
                  'clf__gamma': ['scale', 'auto', 0.01]}
else:
    param_grid = {'clf__hidden_layer_sizes': [(64,32),(128,64,32)],
                  'clf__alpha': [0.0001, 0.001],
                  'clf__learning_rate_init': [0.001, 0.01]}

gs = GridSearchCV(best_model, param_grid, cv=5,
                  scoring='accuracy', n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print('Best params :', gs.best_params_)
print('CV accuracy :', round(gs.best_score_, 4))
print('Test accuracy:', round(accuracy_score(y_test, gs.predict(X_test)), 4))
tuned_model = gs.best_estimator_


## 7. Real-Time Prediction System


In [ ]:
def predict_modulation(snr_db, ber, model=None):
    """Predict optimal modulation for given SNR and BER."""
    if model is None:
        model = tuned_model
    log_ber  = np.log10(np.clip(ber, 1e-10, 1))
    features = np.array([[
        snr_db,
        log_ber,
        snr_db * abs(log_ber),
        snr_db ** 2,
        abs(log_ber)
    ]])
    pred_idx   = model.predict(features)[0]
    probs      = model.predict_proba(features)[0]
    modulation = le.inverse_transform([pred_idx])[0]
    eff_map    = {'BPSK': 1, 'QPSK': 2, '16-QAM': 4, '64-QAM': 6}
    return {
        'modulation'         : modulation,
        'confidence'         : float(probs[pred_idx]),
        'spectral_efficiency': eff_map[modulation],
        'probabilities'      : dict(zip(le.classes_, probs))
    }

# Demo sweep
header = '{:>9} {:>10} {:>12} {:>12} {:>9}'.format('SNR(dB)','BER','Modulation','Confidence','bits/sym')
print(header)
print('-' * 57)
for snr, ber in [(-3,0.12),(4,0.02),(8,3e-3),(15,1e-4),(25,1e-6),(33,1e-7)]:
    r = predict_modulation(snr, ber)
    row = '{:>9.1f} {:>10.2e} {:>12} {:>11.1%} {:>9}'.format(
        snr, ber, r['modulation'], r['confidence'], r['spectral_efficiency'])
    print(row)

# Probability bar chart for one sample
snr_t, ber_t = 15, 1e-4
res = predict_modulation(snr_t, ber_t)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Prediction: SNR=' + str(snr_t) + ' dB, BER=1e-4', fontweight='bold')
mods  = list(res['probabilities'].keys())
probs = list(res['probabilities'].values())
bcol  = ['#E74C3C' if m == res['modulation'] else '#BDC3C7' for m in mods]
bars  = ax1.bar(mods, probs, color=bcol, edgecolor='white', linewidth=1.5)
ax1.set_ylabel('Probability'); ax1.set_ylim([0, 1.1])
ax1.set_title('Class Probabilities')
for bar, p in zip(bars, probs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
             '{:.1%}'.format(p), ha='center', fontsize=10, fontweight='bold')
ax2.pie(probs, labels=mods, autopct='%1.1f%%',
        colors=['#E74C3C','#F39C12','#2ECC71','#3498DB'],
        startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
ax2.set_title('Decision: ' + res['modulation'])
plt.tight_layout(); plt.show()


## 8. Channel Simulation - Adaptive Modulation Over Time


In [ ]:
np.random.seed(0)
T = 500
t = np.arange(T)

snr_traj = (15
            + 12 * np.sin(2 * np.pi * t / 200)
            + 4  * np.random.randn(T)
            + 5  * (np.random.rand(T) < 0.02) * np.random.randn(T) * 10)
ber_traj = np.clip(ber_bpsk(snr_traj)*(1+0.2*np.abs(np.random.randn(T))), 1e-8, 0.5)
mod_traj = [predict_modulation(snr_traj[i], ber_traj[i])['modulation'] for i in range(T)]
eff_map  = {'BPSK':1,'QPSK':2,'16-QAM':4,'64-QAM':6}
eff_traj = [eff_map[m] for m in mod_traj]

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig.suptitle('Adaptive Modulation - Channel Simulation', fontsize=13, fontweight='bold')

axes[0].plot(t, snr_traj, color='#3498DB', lw=1.2)
axes[0].fill_between(t, snr_traj, alpha=0.15, color='#3498DB')
for th, col, lbl in [(5,'#E74C3C','5 dB'),(10,'#F39C12','10 dB'),(20,'#2ECC71','20 dB')]:
    axes[0].axhline(th, color=col, ls='--', lw=1, label='SNR='+lbl)
axes[0].set_ylabel('SNR (dB)'); axes[0].set_title('Channel SNR')
axes[0].legend(loc='upper right', fontsize=8); axes[0].grid(alpha=0.3)

CMAP = {'BPSK':'#E74C3C','QPSK':'#F39C12','16-QAM':'#2ECC71','64-QAM':'#3498DB'}
prev, seg = mod_traj[0], 0
for i in range(1, T+1):
    curr = mod_traj[i] if i < T else None
    if curr != prev or i == T:
        axes[1].axvspan(seg, i-1, facecolor=CMAP[prev], alpha=0.6)
        seg, prev = i, curr
handles = [mpatches.Patch(color=CMAP[m], label=m, alpha=0.7)
           for m in ['BPSK','QPSK','16-QAM','64-QAM']]
axes[1].legend(handles=handles, loc='upper right', fontsize=9)
axes[1].set_yticks([]); axes[1].set_title('Selected Modulation')

axes[2].fill_between(t, eff_traj, step='mid', alpha=0.5, color='#8E44AD')
axes[2].step(t, eff_traj, where='mid', color='#6C3483', lw=1.5)
axes[2].set_ylabel('bits/symbol'); axes[2].set_xlabel('Time Step')
axes[2].set_yticks([1,2,4,6])
axes[2].set_yticklabels(['1 (BPSK)','2 (QPSK)','4 (16-QAM)','6 (64-QAM)'])
axes[2].set_title('Spectral Efficiency'); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

cnt = Counter(mod_traj)
print('Modulation usage:')
for m in ['BPSK','QPSK','16-QAM','64-QAM']:
    print('  {:>8}: {:>4} steps ({:5.1f}%)'.format(m, cnt[m], cnt[m]/T*100))
print('\nAverage spectral efficiency: {:.2f} bits/symbol'.format(np.mean(eff_traj)))


## 9. Cross-Validation & Final Summary


In [ ]:
X_all = np.vstack([X_train, X_val, X_test])
y_all = np.concatenate([y_train, y_val, y_test])

cv_res = {}
print('{:<22} {:>8} {:>8}'.format('Model','Mean','Std'))
print('-' * 42)
for name, pipe in trained_models.items():
    sc = cross_val_score(pipe, X_all, y_all, cv=5, scoring='accuracy', n_jobs=-1)
    cv_res[name] = sc
    print('{:<22} {:>8.4f} {:>8.4f}'.format(name, sc.mean(), sc.std()))

fig, ax = plt.subplots(figsize=(9, 5))
vp = ax.violinplot([cv_res[m] for m in trained_models],
                   positions=range(len(trained_models)),
                   showmeans=True, showmedians=True)
for body in vp['bodies']: body.set_alpha(0.7)
ax.set_xticks(range(len(trained_models)))
ax.set_xticklabels(['RF','GB','SVM','MLP'])
ax.set_ylabel('Accuracy'); ax.set_title('5-Fold CV Accuracy')
ax.set_ylim([0.93, 1.01]); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## 10. Save Model


In [ ]:
import joblib

os.makedirs('adaptive_modulation_model', exist_ok=True)
joblib.dump(tuned_model, 'adaptive_modulation_model/model.pkl')
joblib.dump(le,          'adaptive_modulation_model/label_encoder.pkl')

meta = {
    'project'      : 'ML-Based Adaptive Modulation',
    'author'       : 'Francesco De Florence',
    'best_model'   : best_name,
    'features'     : FEATURES,
    'classes'      : list(le.classes_),
    'test_accuracy': results[best_name]['test']
}
with open('adaptive_modulation_model/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:', os.listdir('adaptive_modulation_model'))

m2 = joblib.load('adaptive_modulation_model/model.pkl')
r  = predict_modulation(18, 2e-5, model=m2)
print('Load check - SNR=18dB, BER=2e-5:', r['modulation'],
      '({:.1%}, {} bits/sym)'.format(r['confidence'], r['spectral_efficiency']))
print('Project complete!')
